# PAC-3310 Calcium Flux Analysis — Consolidated

This notebook analyzes PAC-3310 calcium-flux experiments at **M1, M3, and M5**. It includes replicate-level transient fitting, ΔF/F₀ normalization, 4-parameter logistic (4PL) fitting with an F-test against a flat response, publication-style plots, and final supplementary-table exports.

## 1. Setup

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.stats import f as f_dist
from scipy.stats import t as t_dist
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 110,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'mathtext.default': 'regular',
})

TRACE_COLORS = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#f2c80f', '#a65628', '#f781bf']
RECEPTOR_COLORS = {'M1': '#377eb8', 'M3': '#e41a1c', 'M5': '#4daf4a'}
LABEL_SIZE, TICK_SIZE, LEGEND_SIZE = 13, 11, 10

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR = PROJECT_ROOT / 'data' / 'ca_assay' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'ca_assay' / 'processed'
FIGURE_DIR = PROJECT_ROOT / 'figures' / 'ca_assay'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Data directory: {DATA_DIR}')

## 2. Dataset definitions and corrected assay metadata

In [ ]:
DATASETS = {
    'Carbachol dose titration (M1)': {'file': 'M1_Carbachol_calcium.csv', 'conc_col': 'Carbachol Concentration (log M)', 'control_label': 'Vehicle Control', 'receptor': 'M1', 'mode': 'Carbachol agonism', 'compound': 'Carbachol'},
    'PAC-3310 dose titration (M1)': {'file': 'M1_PAC-3310_calcium.csv', 'conc_col': 'PAC-3310 Concentration (log M)', 'control_label': 'Vehicle Control', 'receptor': 'M1', 'mode': 'PAC-3310 agonism', 'compound': 'PAC-3310'},
    'PAC-3310 + 500 nM carbachol (M1)': {'file': 'M1_PAC-3310_plus_500nM_carbachol_calcium.csv', 'conc_col': 'PAC-3310 Concentration (log M)', 'control_label': 'Carbachol-only Control (500 nM)', 'receptor': 'M1', 'mode': 'PAC-3310 antagonism', 'compound': 'PAC-3310', 'fixed_carbachol_nm': 500, 'control_source': 'PAC-3310 dose titration (M1)'},
    'Carbachol dose titration (M3)': {'file': 'M3_Carbachol_calcium.csv', 'conc_col': 'Carbachol Concentration (log M)', 'control_label': 'Vehicle Control', 'receptor': 'M3', 'mode': 'Carbachol agonism', 'compound': 'Carbachol'},
    'PAC-3310 dose titration (M3)': {'file': 'M3_PAC-3310_calcium.csv', 'conc_col': 'PAC-3310 Concentration (log M)', 'control_label': 'Vehicle Control', 'receptor': 'M3', 'mode': 'PAC-3310 agonism', 'compound': 'PAC-3310'},
    'PAC-3310 + 100 nM carbachol (M3)': {'file': 'M3_PAC-3310_plus_100nM_carbachol_calcium.csv', 'conc_col': 'PAC-3310 Concentration (log M)', 'control_label': 'Carbachol-only Control (100 nM)', 'receptor': 'M3', 'mode': 'PAC-3310 antagonism', 'compound': 'PAC-3310', 'fixed_carbachol_nm': 100, 'control_source': 'PAC-3310 dose titration (M3)'},
    'Carbachol dose titration (M5)': {'file': 'M5_Carbachol_calcium.csv', 'conc_col': 'Carbachol Concentration (log M)', 'control_label': 'Vehicle Control', 'receptor': 'M5', 'mode': 'Carbachol agonism', 'compound': 'Carbachol'},
    'PAC-3310 dose titration (M5)': {'file': 'M5_PAC-3310_calcium.csv', 'conc_col': 'PAC-3310 Concentration (log M)', 'control_label': 'Vehicle Control', 'receptor': 'M5', 'mode': 'PAC-3310 agonism', 'compound': 'PAC-3310'},
    'PAC-3310 + 500 nM carbachol (M5)': {'file': 'M5_PAC-3310_plus_500nM_carbachol_calcium.csv', 'conc_col': 'PAC-3310 Concentration (log M)', 'control_label': 'Carbachol-only Control (500 nM)', 'receptor': 'M5', 'mode': 'PAC-3310 antagonism', 'compound': 'PAC-3310', 'fixed_carbachol_nm': 500, 'control_source': 'PAC-3310 dose titration (M5)'},
}

all_data = {}
validation_rows = []
for name, info in DATASETS.items():
    path = DATA_DIR / info['file']
    df = pd.read_csv(path)
    all_data[name] = df
    conc_col = info['conc_col']
    well_counts = df.groupby([conc_col, 'Replicate']).size()
    validation_rows.append({
        'Condition': name,
        'Rows': len(df),
        'Concentrations including control': df[conc_col].nunique(),
        'Replicates': df['Replicate'].nunique(),
        'Time min (s)': df['Time (s)'].min(),
        'Time max (s)': df['Time (s)'].max(),
        'Points/well min': int(well_counts.min()),
        'Points/well max': int(well_counts.max()),
        'Missing fluorescence': int(df['Fluorescence (RFU)'].isna().sum()),
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)
assert validation_df['Missing fluorescence'].sum() == 0
assert validation_df['Concentrations including control'].eq(8).all()
assert validation_df['Replicates'].eq(4).all()

## 3. Fluorescence traces

Traces are plotted as mean ± SEM across replicate wells. Interpolation to a common five-second grid is used only for visualization; native time points are retained in all exported supplementary tables and in the replicate-level transient fits.

In [ ]:
def ordered_concentrations(df, conc_col, control_label):
    labels = df[conc_col].unique().tolist()
    numeric = sorted([label for label in labels if label != control_label], key=float, reverse=True)
    return numeric + [control_label]


def plot_traces(df, condition_name, info, ax):
    conc_col = info['conc_col']
    control_label = info['control_label']
    for index, conc_label in enumerate(ordered_concentrations(df, conc_col, control_label)):
        subset = df[df[conc_col].eq(conc_label)]
        common_time = np.arange(subset['Time (s)'].min(), subset['Time (s)'].max() + 1, 5)
        traces = []
        for replicate in sorted(subset['Replicate'].unique()):
            rep = subset[subset['Replicate'].eq(replicate)].sort_values('Time (s)')
            traces.append(np.interp(common_time, rep['Time (s)'], rep['Fluorescence (RFU)']))
        traces = np.asarray(traces)
        mean = traces.mean(axis=0)
        sem = traces.std(axis=0, ddof=1) / np.sqrt(len(traces))
        label = f'{float(conc_label):.0f} log M' if conc_label != control_label else control_label
        ax.plot(common_time, mean, lw=1.7, color=TRACE_COLORS[index], label=label)
        ax.fill_between(common_time, mean - sem, mean + sem, color=TRACE_COLORS[index], alpha=0.17)
    ax.set_title(condition_name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Time (s)', fontsize=10)
    ax.set_ylabel('Fluorescence (RFU)', fontsize=10)
    ax.tick_params(labelsize=9)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=7, frameon=False, ncol=2)


fig, axes = plt.subplots(3, 3, figsize=(17, 13))
for ax, (name, info) in zip(axes.flat, DATASETS.items()):
    plot_traces(all_data[name], name, info, ax)
fig.suptitle('PAC-3310 calcium-flux fluorescence traces', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
trace_figure = FIGURE_DIR / 'PAC-3310_calcium_flux_traces.png'
plt.savefig(trace_figure, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {trace_figure}')

## 4. Replicate-level calcium-transient fitting

Each fluorescence trace is modeled as:

$$F(t)=F_{baseline}+A\left(e^{-t/\tau_{decay}}-e^{-t/\tau_{rise}}\right)$$

Each replicate well is fitted independently. If the nonlinear fit fails, the observed maximum is used as a raw-peak fallback. Fit method and R² are retained for auditability.

In [ ]:
def rise_decay_model(t, baseline, amplitude, tau_rise, tau_decay):
    return baseline + amplitude * (np.exp(-t / tau_decay) - np.exp(-t / tau_rise))


def analytical_peak(baseline, amplitude, tau_rise, tau_decay):
    if tau_rise >= tau_decay or tau_rise <= 0 or tau_decay <= 0:
        return 0.0, baseline
    peak_time = (tau_rise * tau_decay / (tau_decay - tau_rise)) * np.log(tau_decay / tau_rise)
    return peak_time, rise_decay_model(peak_time, baseline, amplitude, tau_rise, tau_decay)


def fit_calcium_transient(time, fluorescence):
    time = np.asarray(time, dtype=float)
    fluorescence = np.asarray(fluorescence, dtype=float)
    try:
        baseline_init = fluorescence[:5].mean()
        peak_index = int(np.argmax(fluorescence))
        amplitude_init = fluorescence[peak_index] - baseline_init
        if amplitude_init <= 0:
            raise ValueError('No positive transient')
        tau_rise_init = time[peak_index] / 3 if peak_index > 0 else 1.0
        tau_decay_init = (time[-1] - time[peak_index]) / 2 if peak_index < len(time) - 1 else 10.0
        popt, _ = curve_fit(
            rise_decay_model, time, fluorescence,
            p0=[baseline_init, amplitude_init, tau_rise_init, tau_decay_init],
            bounds=([0, 0, 0.01, 0.1], [np.inf, np.inf, 100, 500]),
            maxfev=5000,
        )
        fitted = rise_decay_model(time, *popt)
        ss_res = np.sum((fluorescence - fitted) ** 2)
        ss_tot = np.sum((fluorescence - fluorescence.mean()) ** 2)
        r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        peak_time, peak_value = analytical_peak(*popt)
        return {
            'Method': 'rise-decay fit', 'Baseline': popt[0], 'Amplitude': popt[1],
            'Tau Rise (s)': popt[2], 'Tau Decay (s)': popt[3],
            'Peak Time (s)': peak_time, 'Peak Fluorescence (RFU)': peak_value,
            'Transient R²': r_squared,
        }
    except Exception:
        peak_index = int(np.argmax(fluorescence))
        baseline = fluorescence.mean()
        return {
            'Method': 'raw peak fallback', 'Baseline': baseline,
            'Amplitude': fluorescence[peak_index] - baseline,
            'Tau Rise (s)': np.nan, 'Tau Decay (s)': np.nan,
            'Peak Time (s)': time[peak_index],
            'Peak Fluorescence (RFU)': fluorescence[peak_index],
            'Transient R²': np.nan,
        }


fit_records = []
fit_lookup = {}
for condition, info in DATASETS.items():
    df = all_data[condition]
    conc_col = info['conc_col']
    fit_lookup[condition] = {}
    for (concentration, replicate), trace in df.groupby([conc_col, 'Replicate'], sort=False):
        trace = trace.sort_values('Time (s)')
        result = fit_calcium_transient(trace['Time (s)'], trace['Fluorescence (RFU)'])
        fit_lookup[condition][(str(concentration), int(replicate))] = result
        fit_records.append({'Condition': condition, 'Receptor': info['receptor'], 'Assay Mode': info['mode'], 'Concentration': concentration, 'Replicate': int(replicate), **result})

transient_fits_df = pd.DataFrame(fit_records)
fit_qc = transient_fits_df.groupby(['Condition', 'Method']).size().unstack(fill_value=0)
fit_qc['Median transient R²'] = transient_fits_df.groupby('Condition')['Transient R²'].median()
display(fit_qc)

## 5. ΔF/F₀ normalization and replicate-level CRC data

$$\Delta F/F_0=\frac{F_{peak}-F_0}{F_0}$$

For each agonism series, F₀ is the grand mean fluorescence of its own vehicle-control trace. For antagonism, F₀ comes from the matching receptor's **PAC-3310 agonism vehicle control**. The fixed-carbachol row H is retained separately as the carbachol-only response reference.

In [ ]:
def control_f0(condition):
    info = DATASETS[condition]
    df = all_data[condition]
    return df.loc[df[info['conc_col']].eq(info['control_label']), 'Fluorescence (RFU)'].mean()


f0_values = {}
crc_records = []
control_response_records = []

for condition, info in DATASETS.items():
    source_condition = info.get('control_source', condition)
    f0 = control_f0(source_condition)
    f0_values[condition] = f0
    f0_description = f"{source_condition} / {DATASETS[source_condition]['control_label']}"
    control_label = info['control_label']

    for (concentration, replicate), result in fit_lookup[condition].items():
        response = (result['Peak Fluorescence (RFU)'] - f0) / f0
        if concentration == control_label:
            control_response_records.append({
                'Condition': condition, 'Receptor': info['receptor'], 'Assay Mode': info['mode'],
                'Control Label': control_label, 'Replicate': replicate,
                'Peak Fluorescence (RFU)': result['Peak Fluorescence (RFU)'],
                'F0 (RFU)': f0, 'F0 Source': f0_description, 'Control ΔF/F0': response,
            })
            continue
        log_conc = float(concentration)
        crc_records.append({
            'Condition': condition, 'Receptor': info['receptor'], 'Assay Mode': info['mode'],
            'Fixed Carbachol (nM)': info.get('fixed_carbachol_nm', np.nan),
            'Concentration (log M)': log_conc, 'Concentration (M)': 10 ** log_conc,
            'Replicate': replicate, 'Peak Fluorescence (RFU)': result['Peak Fluorescence (RFU)'],
            'F0 (RFU)': f0, 'F0 Source': f0_description, 'ΔF/F0': response,
            'Transient Fit Method': result['Method'], 'Transient R²': result['Transient R²'],
        })

crc_df = pd.DataFrame(crc_records)
control_responses_df = pd.DataFrame(control_response_records)

# Express PAC-3310 agonism as a percentage of the maximum observed
# mean carbachol response for the matching receptor.
activation_endpoint = 'PAC-3310 Activation (% Max Carbachol)'
crc_df[activation_endpoint] = np.nan
for receptor in ['M1', 'M3', 'M5']:
    carbachol_condition = f'Carbachol dose titration ({receptor})'
    pac_condition = f'PAC-3310 dose titration ({receptor})'
    max_carbachol_mean = (
        crc_df.loc[crc_df['Condition'].eq(carbachol_condition)]
        .groupby('Concentration (log M)')['ΔF/F0']
        .mean()
        .max()
    )
    pac_mask = crc_df['Condition'].eq(pac_condition)
    crc_df.loc[pac_mask, activation_endpoint] = 100 * crc_df.loc[pac_mask, 'ΔF/F0'] / max_carbachol_mean

# Control-referenced and range-referenced antagonism normalizations.
crc_df['Control-referenced % Inhibition'] = np.nan
crc_df['Range-referenced % Inhibition'] = np.nan
for condition, info in DATASETS.items():
    if info['mode'] != 'PAC-3310 antagonism':
        continue
    mask = crc_df['Condition'].eq(condition)
    carbachol_only = control_responses_df.loc[control_responses_df['Condition'].eq(condition), 'Control ΔF/F0'].mean()
    max_tested_mean = crc_df.loc[mask].groupby('Concentration (log M)')['ΔF/F0'].mean().max()
    crc_df.loc[mask, 'Control-referenced % Inhibition'] = 100 * (1 - crc_df.loc[mask, 'ΔF/F0'] / carbachol_only)
    crc_df.loc[mask, 'Range-referenced % Inhibition'] = 100 * (1 - crc_df.loc[mask, 'ΔF/F0'] / max_tested_mean)

display(crc_df.groupby(['Condition', 'Concentration (log M)'])['ΔF/F0'].agg(['mean', 'sem', 'count']).round(4))
display(control_responses_df.groupby('Condition')['Control ΔF/F0'].agg(['mean', 'sem', 'count']).round(4))

## 6. 4PL curve fitting and F-test model selection

A 4PL model is fitted in log-concentration space. It is retained only when an F-test against a flat response is significant at **α = 0.01**. Otherwise the plot shows data points without a fitted curve and the potency parameters are reported as not estimable.

- Carbachol agonism endpoint: ΔF/F₀
- PAC-3310 agonism endpoint: percentage of the matching receptor's maximum observed mean carbachol ΔF/F₀
- Antagonism endpoint: range-referenced percentage inhibition, calculated relative to the maximum mean response across the tested antagonism series. The unscaled ΔF/F₀ and explicit carbachol-control-referenced percentage are also exported.

In [ ]:
def model_4pl(log_conc, bottom, hill_slope, log_ec50, top):
    return bottom + (top - bottom) / (1 + 10 ** ((log_ec50 - log_conc) * hill_slope))


def fit_4pl(log_concentrations, responses):
    x = np.asarray(log_concentrations, dtype=float)
    y = np.asarray(responses, dtype=float)
    try:
        popt, pcov = curve_fit(
            model_4pl, x, y,
            p0=[y.min(), 1.0, np.median(x), y.max()],
            bounds=([-np.inf, 0.01, x.min() - 2, -np.inf], [np.inf, 50, x.max() + 2, np.inf]),
            maxfev=10000,
        )
        predicted = model_4pl(x, *popt)
        ss_res = np.sum((y - predicted) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        log_ec50_se = np.sqrt(pcov[2, 2]) if np.isfinite(pcov[2, 2]) and pcov[2, 2] > 0 else np.nan
        if np.isfinite(log_ec50_se) and len(y) > 4:
            t_critical = t_dist.ppf(0.975, df=len(y) - 4)
            ci = (10 ** (popt[2] - t_critical * log_ec50_se), 10 ** (popt[2] + t_critical * log_ec50_se))
        else:
            ci = (np.nan, np.nan)
        return {
            'popt': popt, 'Bottom': popt[0], 'Hill Slope': popt[1],
            'logEC50': popt[2], 'Top': popt[3], 'EC50 (M)': 10 ** popt[2],
            'EC50 95% CI Low (M)': ci[0], 'EC50 95% CI High (M)': ci[1],
            'EC80 (M)': 10 ** (popt[2] + np.log10(4) / abs(popt[1])),
            'R²': r_squared, 'SS Residual': ss_res,
        }
    except Exception:
        return None


def f_test_vs_flat(responses, fit_result, alpha=0.01):
    y = np.asarray(responses, dtype=float)
    ss_null = np.sum((y - y.mean()) ** 2)
    ss_model = fit_result['SS Residual']
    df_model = len(y) - 4
    if df_model <= 0 or ss_model <= 0:
        return np.nan, 1.0, False
    f_statistic = ((ss_null - ss_model) / 3) / (ss_model / df_model)
    p_value = 1 - f_dist.cdf(f_statistic, 3, df_model)
    return f_statistic, p_value, bool(p_value < alpha)


curve_results = {}
summary_records = []
for condition, info in DATASETS.items():
    subset = crc_df[crc_df['Condition'].eq(condition)].copy()
    if info['mode'] == 'PAC-3310 antagonism':
        endpoint = 'Range-referenced % Inhibition'
    elif info['mode'] == 'PAC-3310 agonism':
        endpoint = activation_endpoint
    else:
        endpoint = 'ΔF/F0'
    x = subset['Concentration (log M)'].to_numpy(float)
    y = subset[endpoint].to_numpy(float)
    result = fit_4pl(x, y)
    if result is None:
        significant, f_statistic, p_value = False, np.nan, np.nan
    else:
        f_statistic, p_value, significant = f_test_vs_flat(y, result)
    if result is not None:
        direction = 'increasing' if result['Top'] > result['Bottom'] else 'decreasing'
        potency_in_range = bool(x.min() <= result['logEC50'] <= x.max())
        expected_direction = direction == 'increasing'
    else:
        direction = 'not estimable'
        potency_in_range = False
        expected_direction = False
    interpretable = bool(significant and potency_in_range and expected_direction)
    if interpretable:
        model_label = '4PL'
        interpretation = 'Concentration-dependent response with in-range potency'
    elif significant:
        reasons = []
        if not potency_in_range:
            reasons.append('potency outside tested range')
        if not expected_direction:
            reasons.append('opposite response direction')
        model_label = '4PL trend (not a valid potency estimate)'
        interpretation = '; '.join(reasons)
    else:
        model_label = 'No significant 4PL'
        interpretation = 'No concentration-dependent response at α=0.01'
    curve_results[condition] = {
        'endpoint': endpoint, 'fit': result, 'significant': significant,
        'interpretable': interpretable, 'potency_in_range': potency_in_range,
        'expected_direction': expected_direction, 'F': f_statistic, 'p': p_value,
    }
    control_mean = control_responses_df.loc[control_responses_df['Condition'].eq(condition), 'Control ΔF/F0'].mean()
    summary_records.append({
        'Condition': condition, 'Receptor': info['receptor'], 'Assay Mode': info['mode'],
        'Fixed Carbachol (nM)': info.get('fixed_carbachol_nm', np.nan),
        'Endpoint': endpoint, 'Model': model_label,
        'EC50 (M)': result['EC50 (M)'] if interpretable else np.nan,
        'EC50 (µM)': result['EC50 (M)'] * 1e6 if interpretable else np.nan,
        'EC50 95% CI Low (M)': result['EC50 95% CI Low (M)'] if interpretable else np.nan,
        'EC50 95% CI High (M)': result['EC50 95% CI High (M)'] if interpretable else np.nan,
        'EC80 (M)': result['EC80 (M)'] if interpretable else np.nan,
        'Hill Slope': result['Hill Slope'] if interpretable else np.nan,
        'Bottom': result['Bottom'] if interpretable else np.nan,
        'Top': result['Top'] if interpretable else np.nan,
        'R²': result['R²'] if result and significant else np.nan,
        'F Statistic': f_statistic, 'F-test p': p_value,
        'Curve Direction': direction if significant else 'not significant',
        'Candidate logEC50': result['logEC50'] if result and significant else np.nan,
        'Potency Within Tested Range': potency_in_range if significant else False,
        'Interpretation': interpretation,
        'Carbachol-only Control Mean ΔF/F0': control_mean if info['mode'] == 'PAC-3310 antagonism' else np.nan,
    })

fit_summary_df = pd.DataFrame(summary_records)
display(fit_summary_df)

## 7. Consolidated CRC panels

In [ ]:
def plot_crc(condition, ax):
    info = DATASETS[condition]
    subset = crc_df[crc_df['Condition'].eq(condition)]
    endpoint = curve_results[condition]['endpoint']
    grouped = subset.groupby('Concentration (log M)')[endpoint]
    means = grouped.mean().sort_index()
    sems = grouped.sem().sort_index()
    color = RECEPTOR_COLORS[info['receptor']]
    ax.errorbar(means.index, means.values, yerr=sems.values, fmt='o', color=color,
                markersize=5.5, elinewidth=1.3, markeredgecolor='white', markeredgewidth=0.5,
                label=info['receptor'], zorder=5)
    result = curve_results[condition]
    if result['interpretable']:
        x_fit = np.linspace(means.index.min(), means.index.max(), 250)
        ax.plot(x_fit, model_4pl(x_fit, *result['fit']['popt']), color=color, lw=2)
    # A zero-reference line is useful for the raw-response and
    # inhibition panels, but it would force the activation panels'
    # automatic lower limit down to zero.
    if info['mode'] != 'PAC-3310 agonism':
        ax.axhline(0, color='0.5', lw=0.8, alpha=0.5)
    ax.set_title(condition, fontsize=10.5, fontweight='bold')
    ax.set_xlabel(f"{info['compound']} concentration (log M)", fontsize=10)
    ylabel = 'Activation (% max carbachol response)' if info['mode'] == 'PAC-3310 agonism' else endpoint
    ax.set_ylabel(ylabel, fontsize=10)
    ax.tick_params(labelsize=9)
    ax.grid(True, alpha=0.25)


fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, (condition, info) in zip(axes.flat, DATASETS.items()):
    plot_crc(condition, ax)

for row in range(3):
    axes[row, 1].set_ylim(bottom=-5, top=100)
    axes[row, 2].set_ylim(top=100)

fig.suptitle('PAC-3310 calcium-flux concentration-response analysis', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
crc_figure = FIGURE_DIR / 'PAC-3310_calcium_flux_CRCs.png'
plt.savefig(crc_figure, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {crc_figure}')

## 8. Publication-style cross-receptor comparison

In [ ]:
mode_order = ['Carbachol agonism', 'PAC-3310 agonism', 'PAC-3310 antagonism']
panel_titles = ['Carbachol agonism', 'PAC-3310 agonism', 'PAC-3310 + carbachol antagonism']
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))

for ax, mode, title in zip(axes, mode_order, panel_titles):
    for receptor in ['M1', 'M3', 'M5']:
        condition = next(name for name, info in DATASETS.items() if info['mode'] == mode and info['receptor'] == receptor)
        subset = crc_df[crc_df['Condition'].eq(condition)]
        endpoint = curve_results[condition]['endpoint']
        grouped = subset.groupby('Concentration (log M)')[endpoint]
        means, sems = grouped.mean().sort_index(), grouped.sem().sort_index()
        color = RECEPTOR_COLORS[receptor]
        label = receptor
        if mode == 'PAC-3310 antagonism':
            fixed = int(DATASETS[condition]['fixed_carbachol_nm'])
            label = f'{receptor} ({fixed} nM carbachol)'
        ax.errorbar(means.index, means.values, yerr=sems.values, fmt='o', color=color,
                    markersize=5.5, elinewidth=1.3, markeredgecolor='white', markeredgewidth=0.5,
                    label=label, zorder=5)
        result = curve_results[condition]
        if result['interpretable']:
            x_fit = np.linspace(means.index.min(), means.index.max(), 250)
            ax.plot(x_fit, model_4pl(x_fit, *result['fit']['popt']), color=color, lw=2)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Concentration (log M)', fontsize=LABEL_SIZE)
    if mode == 'PAC-3310 antagonism':
        ylabel = 'Range-referenced % inhibition'
    elif mode == 'PAC-3310 agonism':
        ylabel = 'Activation (% max carbachol response)'
    else:
        ylabel = r'$\Delta$F/F$_0$'
    ax.set_ylabel(ylabel, fontsize=LABEL_SIZE)
    ax.tick_params(labelsize=TICK_SIZE)
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False, fontsize=LEGEND_SIZE)

axes[1].set_ylim(bottom=-5, top=100)
axes[2].set_ylim(top=100)

plt.tight_layout()
publication_figure = FIGURE_DIR / 'PAC-3310_calcium_flux_publication_panels.png'
plt.savefig(publication_figure, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {publication_figure}')

## 9. Export analysis tables

In [ ]:
transient_fits_path = OUTPUT_DIR / 'PAC-3310_calcium_flux_transient_fits.csv'
crc_path = OUTPUT_DIR / 'PAC-3310_calcium_flux_CRC_replicates.csv'
controls_path = OUTPUT_DIR / 'PAC-3310_calcium_flux_control_responses.csv'
summary_path = OUTPUT_DIR / 'PAC-3310_calcium_flux_fit_summary.csv'

transient_fits_df.to_csv(transient_fits_path, index=False)
crc_df.to_csv(crc_path, index=False)
control_responses_df.to_csv(controls_path, index=False)
fit_summary_df.to_csv(summary_path, index=False)

for path in [transient_fits_path, crc_path, controls_path, summary_path]:
    print(f'Saved: {path}')

print('\nSignificant 4PL results (F-test α=0.01):')
display(fit_summary_df.loc[fit_summary_df['Model'].eq('4PL'), [
    'Condition', 'Endpoint', 'EC50 (µM)', 'Hill Slope', 'R²', 'F-test p', 'Curve Direction'
]])

## Analysis notes

- Replicate unit: one plate-reader cell-well trace; observed n = 4 wells per dose for every exported condition.
- No fluorescence values were imputed or removed. Unequal trace lengths (51 or 61 native time points) are preserved.
- The 5-second interpolation is used only in trace visualization.
- Non-significant 4PL fits do not receive potency estimates.
- Antagonism percentages are not clamped; negative values indicate responses above the selected reference.